In [ ]:
'''
Keras_rl_cartpole: Keras-RLの基本的な使い方（CartPole課題を例に）

[Note]
* Keras-RLのインストール（Keras-RLのバージョンに注意）
  % pip install keras-rl2
  
* 上記によってkerasとtensorflowもインストールされるはずだが、
  これらのライブラリが見つからないエラーが生じる場合は、コンソールから
  以下のコマンドを入力する
  % pip install keras
  % pip install tensorflow

* 上記をインストール後にカーネルを再起動すること
'''

###################################################################
#
# モジュールの読み込み
#
###################################################################

#-----------------------------
# Open AI Gym (強化学習環境用ライブラリ)
#-----------------------------
import gym

#-----------------------------
# Keras (Deep Neural Network用ライブラリ)
#-----------------------------
from keras.models import Sequential
from keras.layers import Dense, Activation, Flatten
from keras.optimizer_v1 import Adam # <- optimizerのバージョンに注意

# 現バージョンのエラーを防ぐために以下を書いておく
Adam._name = 'hey'

#-----------------------------
# Keras-RL (Kerasを活用した強化学習用ライブラリ)
#-----------------------------

# Keras-RLのモジュールの読み込み
from rl.agents.dqn import DQNAgent # Deep Q Network agentを利用
from rl.policy import BoltzmannQPolicy # Boltzmann方策を利用
from rl.memory import SequentialMemory # Deep Network学習のためのメモリバッファを利用

###################################################################
#
# RL環境の構築
#
###################################################################

# Cart-Pole課題の環境を読み込む
env = gym.make('CartPole-v1')

# 環境の初期化（必ずしも必要ない）
env.reset()

# 選択可能な行動数を nb_actions に格納する（後で利用するため）
nb_actions = env.action_space.n

###################################################################
#
# RLエージェントの構築
#
###################################################################

#-----------------------------
# 価値関数表現（近似）用ニューラルネットワークの定義（Deep Q networkの設定）
#-----------------------------

# Sequentialインスタンスでオブジェクト作成
model = Sequential()
# 観測空間に次元を合わせて入力層を追加
model.add(Flatten(input_shape=(1,) + env.observation_space.shape))
# 第1隠れ層を追加（全結合でReLU型活性関数を利用）
model.add(Dense(16, activation='relu'))
# 第2隠れ層を追加（全結合でReLU型活性関数を利用）
model.add(Dense(16, activation='relu'))
# 第3隠れ層を追加（全結合でReLU型活性関数を利用）
model.add(Dense(16, activation='relu'))
# 出力層を追加（ユニット数=行動数、線形活性関数の利用）となっていることに注意
model.add(Dense(env.action_space.n, activation='linear'))

#-----------------------------
# 強化学習エージェント用メタヒューリスティック（個性）の設定
#-----------------------------

# Deep Q networkの学習データ用メモリ（Experience replayバッファ）の設定
# limit: 最大学習データサンプル数, window_length: 過去何タイムステップまでの観測を状態とするか, continuous課題か否か)
memory = SequentialMemory(limit=100, window_length=1, ignore_episode_boundaries=False)

# Policyの設定
policy = BoltzmannQPolicy()


#-----------------------------
# 強化学習エージェントを構築
#-----------------------------

# DQNAgentクラスで強化学習エージェントを構築
agent = DQNAgent(model=model, nb_actions=env.action_space.n, memory=memory, policy=policy, nb_steps_warmup=10, target_model_update=1e-2)
# nb_actions: 選択可能な行動数
# nb_steps_warmup: ネットワークの学習前に何試行行うか(burn-inのための試行回数)
# target_model_update: 0-1の場合 -> soft_update; 1以上の場合 -> target_model_update試行回数ごとにパラメータの更新

# Agent内のニューラルネットワークの最適化をAdam法に設定（誤差評価はMAE (mean absolute error)）
agent.compile(Adam(lr=1e-3), metrics=['mae'])

###################################################################
#
# 環境とインタラクションしながら強化学習を実行
#
###################################################################

history = agent.fit(env, nb_steps=1000, visualize=False, verbose=1, log_interval=500)
# nb_steps: 総試行回数の設定
# 課題の状態を可視化するか
# verbose: 0 -> 途中経過を表示しない; 1 -> log_intervalで設定された試行回数ごとに表示; 2 -> 1エピソードごとに表示


In [ ]:
###################################################################
#
# 学習曲線の表示
#
###################################################################

import numpy as np
import matplotlib.pyplot as plt
plt.plot(np.arange(len(history.history["episode_reward"])),history.history["episode_reward"],label="ddqn")
plt.xlabel('Number of episodes')
plt.ylabel('Cumulative reward per episode')
plt.show()

###################################################################
#
# 学習後のモデルパラメータの保存
#
###################################################################

agent.save_weights('ddqn_{}_weights.h5f'.format('CartPole-v0'), overwrite=True)
# 読み込みは agent.load_weights('ファイル名')で可能

###################################################################
#
# 学習後の方策を環境でテスト
#
###################################################################

agent.test(env, nb_episodes=10, visualize=True)